# Ensemble Image Anomaly Detection

## 1. Image Feature Extraction

**Purpose:** Convert a directory of images into quantitative features that can be used for downstream anomaly detection.

**Required input:** Update the image directory path in the first code cell after the library imports. Confirm that all images to be analyzed are stored in that directory before running the notebook.

**Process overview:** Each image is loaded, cropped to 70% of its original size, converted to grayscale, and analyzed for blobs or regions of interest. The workflow records shape, size, and intensity statistics for detected blobs, then calculates texture measures such as contrast, energy, and entropy from the image histogram.

**Output:** The resulting table contains blob-level characteristics and whole-image texture features. These features provide the structured input used by the anomaly detection models in the following sections.

In [ ]:
import cv2
import numpy as np
import os
import pandas as pd
from scipy.stats import entropy, kurtosis, skew
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from itertools import product

# Directory containing images (edit as necessary)
image_dir = 'filepath'

# Get list of image files in the directory
image_files = [
    f for f in os.listdir(image_dir)
    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))
]

# List to hold data for all images
data_list = []

# Loop over all images
for image_file in image_files:
    image_path = os.path.join(image_dir, image_file)
    print(f"Processing {image_file}...")

    # Load the color image (in BGR format)
    image = cv2.imread(image_path)
    if image is None:
        print(f"Could not read {image_file}, skipping.")
        continue

    # Crop the image to 70% of its original size, centered and maintaining the same aspect ratio
    height, width = image.shape[:2]
    crop_ratio = 0.7
    new_width = int(width * crop_ratio)
    new_height = int(height * crop_ratio)
    x_start = int((width - new_width) / 2)
    y_start = int((height - new_height) / 2)
    # Crop the image
    cropped_image = image[
        y_start:y_start + new_height, x_start:x_start + new_width
    ]

    # Convert the cropped image to grayscale for further analysis
    gray_image = cv2.cvtColor(cropped_image, cv2.COLOR_BGR2GRAY)

    # Option to apply a simple contrast adjustment (no adjustment at alpha=1 and beta=0)
    alpha = 1  # Contrast control (1.0-3.0)
    beta = 0   # Brightness control (0-100)
    adjusted_image = cv2.convertScaleAbs(gray_image, alpha=alpha, beta=beta)

    # Blob detection setup
    # Set up SimpleBlobDetector parameters.
    params = cv2.SimpleBlobDetector_Params()

    # Adjust thresholds
    params.minThreshold = 0
    params.maxThreshold = 255
    params.thresholdStep = 5  # Smaller step for finer detection

    # Filter by area
    params.filterByArea = True
    params.minArea = 500      # Increased to ignore smaller blobs
    params.maxArea = 1e9      # Increased to include larger blobs

    # Enable filtering by circularity
    params.filterByCircularity = True
    params.minCircularity = 0.1  # Adjusted to include blobs that are not perfect circles

    # Enable filtering by convexity
    params.filterByConvexity = True
    params.minConvexity = 0.5  # Adjusted to include less convex blobs

    # Enable filtering by inertia (elongation)
    params.filterByInertia = True
    params.minInertiaRatio = 0.1  # Adjusted to include elongated blobs

    # Disable filtering by blob color
    params.filterByColor = False

    # Create a detector with the parameters
    detector = cv2.SimpleBlobDetector_create(params)

    # Detect blobs in the adjusted cropped image
    keypoints = detector.detect(adjusted_image)

    # Create a list to store mean grey values of blobs
    mean_grey_values = []

    # Collect diameters and areas of the detected blobs
    diameters = []
    areas = []

    for keypoint in keypoints:
        x, y = keypoint.pt  # Blob centroid
        diameter = keypoint.size  # Size of the blob
        radius = int(diameter / 2)

        # Create a mask for the blob
        mask = np.zeros_like(adjusted_image, dtype=np.uint8)
        cv2.circle(mask, (int(x), int(y)), radius, 255, thickness=-1)  # White filled circle

        # Calculate mean grey value within the blob
        mean_val = cv2.mean(adjusted_image, mask=mask)[0]
        mean_grey_values.append(mean_val)

        # Collect diameters and areas
        diameters.append(diameter)
        area = np.pi * (diameter / 2) ** 2  # Area of the blob
        areas.append(area)

    # Number of blobs detected
    num_blobs = len(keypoints)
    print(f"Number of blobs detected: {num_blobs}")

    # Convert lists to numpy arrays for statistical calculations
    diameters = np.array(diameters)
    areas = np.array(areas)
    mean_grey_values = np.array(mean_grey_values)

    # Initialize data dictionary for this image
    data = {'image file': image_file}

    # Calculate summary statistics if blobs are detected
    if num_blobs > 0:
        # Statistics for diameters
        data['avg_diameter'] = np.mean(diameters)
        data['std_diameter'] = np.std(diameters)
        data['min_diameter'] = np.min(diameters)
        data['max_diameter'] = np.max(diameters)
        data['median_diameter'] = np.median(diameters)

        # Statistics for areas
        data['avg_area'] = np.mean(areas)
        data['std_area'] = np.std(areas)
        data['min_area'] = np.min(areas)
        data['max_area'] = np.max(areas)
        data['median_area'] = np.median(areas)

        # Statistics for mean grey values within blobs
        data['avg_mean_grey_blob'] = np.mean(mean_grey_values)
        data['std_mean_grey_blob'] = np.std(mean_grey_values)
        data['min_mean_grey_blob'] = np.min(mean_grey_values)
        data['max_mean_grey_blob'] = np.max(mean_grey_values)
        data['median_mean_grey_blob'] = np.median(mean_grey_values)
    else:
        # Set to NaN if no blobs detected
        data['avg_diameter'] = np.nan
        data['std_diameter'] = np.nan
        data['min_diameter'] = np.nan
        data['max_diameter'] = np.nan
        data['median_diameter'] = np.nan

        data['avg_area'] = np.nan
        data['std_area'] = np.nan
        data['min_area'] = np.nan
        data['max_area'] = np.nan
        data['median_area'] = np.nan

        data['avg_mean_grey_blob'] = np.nan
        data['std_mean_grey_blob'] = np.nan
        data['min_mean_grey_blob'] = np.nan
        data['max_mean_grey_blob'] = np.nan
        data['median_mean_grey_blob'] = np.nan

    data['num_blobs'] = num_blobs

    # Texture analysis on the cropped image
    # Calculate the mean grey value (visual density) of the entire cropped image
    mean_grey_value = np.mean(adjusted_image)
    data['mean_grey_value_image'] = mean_grey_value

    # Calculate texture features using histogram and simple statistics
    hist = cv2.calcHist([adjusted_image], [0], None, [256], [0, 256])
    hist = hist.flatten()

    # Normalize the histogram to get probabilities
    hist_norm = hist / hist.sum()

    # Contrast: Standard deviation of the pixel values
    contrast = np.std(adjusted_image)
    data['contrast'] = contrast

    # Energy: Sum of squared elements in the normalized histogram
    energy = np.sum(hist_norm ** 2)
    data['energy'] = energy

    # Homogeneity: Calculated using the normalized histogram
    homogeneity = np.sum(hist_norm / (1 + np.arange(256)))
    data['homogeneity'] = homogeneity

    # Entropy: Measure of randomness in the image
    entropy_value = -np.sum(hist_norm * np.log2(hist_norm + 1e-10))
    data['entropy'] = entropy_value

    # Variance: Variance of the pixel values
    variance = np.var(adjusted_image)
    data['variance'] = variance

    # Skewness: Measure of asymmetry of the pixel intensity distribution
    skewness = skew(adjusted_image.flatten())
    data['skewness'] = skewness

    # Kurtosis: Measure of the "tailedness" of the pixel intensity distribution
    kurt = kurtosis(adjusted_image.flatten())
    data['kurtosis'] = kurt

    # Sum of Squares (Variance of the histogram)
    mean_hist_value = np.mean(hist)
    sum_of_squares = np.sum((hist - mean_hist_value) ** 2)
    data['sum_of_squares'] = sum_of_squares

    # Sum Average: Average intensity based on the histogram
    sum_average = np.sum(np.arange(256) * hist_norm)
    data['sum_average'] = sum_average

    # Sum Entropy: Entropy of the sum distribution (same as image entropy here)
    sum_entropy = -np.sum(hist_norm * np.log2(hist_norm + 1e-10))
    data['sum_entropy'] = sum_entropy

    # Difference Entropy: Entropy of the difference distribution
    diff_hist = np.abs(hist[:-1] - hist[1:])
    diff_hist_norm = diff_hist / diff_hist.sum()
    difference_entropy = -np.sum(diff_hist_norm * np.log2(diff_hist_norm + 1e-10))
    data['difference_entropy'] = difference_entropy

    # Autocorrelation: Measure of repeating patterns in the histogram
    autocorr = np.correlate(hist_norm, hist_norm, mode='full')
    autocorr_value = autocorr[autocorr.size // 2]
    data['autocorrelation'] = autocorr_value

    # Append the data for this image to the list
    data_list.append(data)

# Create a DataFrame from the collected data
df = pd.DataFrame(data_list)

# Display the DataFrame
print(df)

# Save to CSV in the same directory as the images
output_csv_path = os.path.join(image_dir, 'image_analysis_results.csv')
df.to_csv(output_csv_path, index=False)
print(f"\nResults saved to {output_csv_path}")

## 2. Isolation Forest Anomaly Detection

**Purpose:** Identify images whose extracted features differ substantially from the overall dataset.

**Process overview:** The workflow selects numerical columns, removes rows with missing values, standardizes the features, and fits an Isolation Forest model. The model flags potentially unusual observations and assigns each image an anomaly score.

**Method note:** Isolation Forest is an unsupervised anomaly detection method that isolates unusual points through random feature splits. Observations that require fewer splits to isolate are more likely to be anomalous. In this notebook, those observations may represent images with atypical visual patterns.

In [ ]:
# Isolation Forest

# Assuming 'df' is your DataFrame from the previous analysis
# First, select numerical features for analysis (exclude 'image file' and any non-numeric columns)
numeric_cols = df.select_dtypes(include=[np.number]).columns
features = df[numeric_cols]

# Handle missing values: Remove rows with NaN values in the features
# Alternatively, you can fill NaNs with the mean or median
features_clean = features.dropna()
df_clean = df.loc[features_clean.index]

# Scale the features using StandardScaler
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features_clean)

# Initialize Isolation Forest
iso_forest = IsolationForest(contamination='auto', random_state=42)
iso_forest.fit(features_scaled)

# Predict anomalies (-1 for anomaly, 1 for normal)
df_clean['Outlier_IsolationForest'] = iso_forest.predict(features_scaled) == -1

# Add anomaly scores (the lower, the more anomalous)
df_clean['Anomaly_Score_IF'] = iso_forest.decision_function(features_scaled)

# Display outliers
outliers_if = df_clean[df_clean['Outlier_IsolationForest']]
print("Outliers detected using Isolation Forest:")
print(outliers_if[['image file', 'Anomaly_Score_IF']])

# Optionally, merge the outlier information back into the original DataFrame
# This will add NaN for rows that were dropped due to NaNs in features
df['Outlier_IsolationForest'] = df_clean['Outlier_IsolationForest']
df['Anomaly_Score_IF'] = df_clean['Anomaly_Score_IF']

## 3. Autoencoder Anomaly Detection

**Purpose:** Detect images that are difficult for a neural network to reconstruct from the learned feature representation.

**Process overview:** The workflow selects numerical features, standardizes them, trains an autoencoder, and compares the reconstructed feature values with the original inputs. Images with reconstruction errors above the selected threshold are flagged as potential outliers.

**Method note:** An autoencoder learns a compressed representation of the data through an encoder, bottleneck, and decoder. Common patterns tend to reconstruct well, while rare or unusual patterns often produce larger reconstruction errors.

In [ ]:
# Autoencoder Neural Network

# Assuming 'df' is your DataFrame from the previous analysis
# First, select numerical features for analysis (exclude 'image file' and any non-numeric columns)
numeric_cols = df.select_dtypes(include=[np.number]).columns
features = df[numeric_cols]

# Handle missing values: Remove rows with NaN values in the features
features_clean = features.dropna()
df_clean = df.loc[features_clean.index]

# Standardize the features using StandardScaler
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features_clean)

# Define the Autoencoder model
input_dim = features_scaled.shape[1]
encoding_dim = int(input_dim / 2)  # You can adjust the encoding dimension

# Build the Autoencoder
input_layer = layers.Input(shape=(input_dim,))
# Encoder
encoded = layers.Dense(encoding_dim, activation='relu')(input_layer)
# Bottleneck layer
bottleneck = layers.Dense(int(encoding_dim / 2), activation='relu')(encoded)
# Decoder
decoded = layers.Dense(encoding_dim, activation='relu')(bottleneck)
output_layer = layers.Dense(input_dim, activation='linear')(decoded)

autoencoder = models.Model(inputs=input_layer, outputs=output_layer)
autoencoder.compile(optimizer='adam', loss='mse')

# Print the model summary
print(autoencoder.summary())

# Train the Autoencoder
# Use EarlyStopping to prevent overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = autoencoder.fit(
    features_scaled, features_scaled,
    epochs=100,
    batch_size=32,
    shuffle=True,
    validation_split=0.1,
    verbose=1,
    callbacks=[early_stopping]
)

# Plot training and validation loss over epochs
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss', color='blue')
plt.plot(history.history['val_loss'], label='Validation Loss', color='orange')
plt.title('Autoencoder Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Compute reconstruction errors
reconstructions = autoencoder.predict(features_scaled)
mse = np.mean(np.power(features_scaled - reconstructions, 2), axis=1)
df_clean['Reconstruction_Error'] = mse

# Set threshold for anomalies (e.g., 95th percentile)
threshold = np.percentile(mse, 95)
print(f"Reconstruction error threshold (95th percentile): {threshold}")

# Identify outliers
df_clean['Outlier_Autoencoder'] = mse > threshold

# Display outliers
outliers_ae = df_clean[df_clean['Outlier_Autoencoder']]
print("Outliers detected using Autoencoder:")
print(outliers_ae[['image file', 'Reconstruction_Error']])

# Optionally, merge the outlier information back into the original DataFrame
df['Reconstruction_Error'] = df_clean['Reconstruction_Error']
df['Outlier_Autoencoder'] = df_clean['Outlier_Autoencoder']

# Visualize reconstruction errors
plt.figure(figsize=(10, 6))
plt.hist(df_clean['Reconstruction_Error'], bins=50, color='blue', alpha=0.7)
plt.axvline(threshold, color='red', linestyle='--', label='Threshold')
plt.title('Histogram of Reconstruction Errors')
plt.xlabel('Reconstruction Error')
plt.ylabel('Frequency')
plt.legend()
plt.show()

## 4. Local Outlier Factor Anomaly Detection

**Purpose:** Detect observations that are sparse or isolated relative to nearby observations in feature space.

**Process overview:** The workflow selects numeric features, handles missing values, standardizes the data, and applies Local Outlier Factor (LOF). Each observation receives an outlier score based on how its local density compares with the density of its neighbors. The score distribution is then visualized.

**Method note:** LOF is well suited for datasets where normal observations may form regions with different densities. Points with much lower local density than their neighbors are more likely to represent unusual image patterns.

In [ ]:
# Local Outlier Factor

# Assuming 'df' is your DataFrame from the previous analysis
# First, select numerical features for analysis (exclude 'image file' and any non-numeric columns)
numeric_cols = df.select_dtypes(include=[np.number]).columns
features = df[numeric_cols]

# Handle missing values: Remove rows with NaN values in the features
features_clean = features.dropna()
df_clean = df.loc[features_clean.index]

# Standardize the features using StandardScaler
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features_clean)

# Initialize Local Outlier Factor
# n_neighbors can be adjusted; default is 20
lof = LocalOutlierFactor(n_neighbors=20, contamination='auto')

# Fit the model and predict outliers
# The fit_predict method returns -1 for outliers and 1 for inliers
y_pred = lof.fit_predict(features_scaled)

# Add LOF scores and outlier labels to the DataFrame
df_clean['LOF_Score'] = -lof.negative_outlier_factor_
df_clean['Outlier_LOF'] = y_pred == -1  # True if outlier

# Display outliers
outliers_lof = df_clean[df_clean['Outlier_LOF']]
print("Outliers detected using Local Outlier Factor:")
print(outliers_lof[['image file', 'LOF_Score']])

# Optionally, merge the outlier information back into the original DataFrame
df['LOF_Score'] = df_clean['LOF_Score']
df['Outlier_LOF'] = df_clean['Outlier_LOF']

# Visualize the distribution of LOF scores
plt.figure(figsize=(10, 6))
plt.hist(df_clean['LOF_Score'], bins=50, color='blue', alpha=0.7)
plt.title('Histogram of LOF Scores')
plt.xlabel('LOF Score')
plt.ylabel('Frequency')
plt.show()

## 5. One-Class SVM Anomaly Detection

**Purpose:** Learn the boundary of typical image-feature behavior and flag observations that fall outside that boundary.

**Process overview:** The workflow selects numeric features, handles missing values, standardizes the data, and trains a One-Class SVM model. The model predicts whether each image is consistent with the learned normal region and assigns an anomaly score based on distance from the decision boundary.

**Method note:** One-Class SVM is useful when anomalies are expected to be relatively rare. Images that fall outside the learned boundary are treated as potential outliers.

In [ ]:
# One-Class Support Vector Machine

# Assuming 'df' is your DataFrame from the previous analysis
# First, select numerical features for analysis (exclude 'image file' and any non-numeric columns)
numeric_cols = df.select_dtypes(include=[np.number]).columns
features = df[numeric_cols]

# Handle missing values: Remove rows with NaN values in the features
features_clean = features.dropna()
df_clean = df.loc[features_clean.index]

# Standardize the features using StandardScaler
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features_clean)

# Initialize the One-Class SVM
# nu: An upper bound on the fraction of training errors (anomalies)
# kernel: 'rbf' is a common choice
oc_svm = OneClassSVM(kernel='rbf', gamma='auto', nu=0.02)

# Fit the model
oc_svm.fit(features_scaled)

# Predict anomalies
# The predict method returns -1 for outliers and 1 for inliers
y_pred = oc_svm.predict(features_scaled)

# Add anomaly scores and outlier labels to the DataFrame
# decision_function returns the distance of the samples to the separating hyperplane
df_clean['OCSVM_Score'] = oc_svm.decision_function(features_scaled)
df_clean['Outlier_OCSVM'] = y_pred == -1  # True if outlier

# Display outliers
outliers_ocsvm = df_clean[df_clean['Outlier_OCSVM']]
print("\nOutliers detected using One-Class SVM:")
print(outliers_ocsvm[['image file', 'OCSVM_Score']])

# Optionally, merge the outlier information back into the original DataFrame
df['OCSVM_Score'] = df_clean['OCSVM_Score']
df['Outlier_OCSVM'] = df_clean['Outlier_OCSVM']

# Visualize the distribution of OCSVM scores
plt.figure(figsize=(10, 6))
plt.hist(df_clean['OCSVM_Score'], bins=50, color='blue', alpha=0.7)
plt.title('Histogram of One-Class SVM Scores')
plt.xlabel('OCSVM Score')
plt.ylabel('Frequency')
plt.show()

## 6. DBSCAN Clustering and Outlier Detection

**Purpose:** Cluster images with similar feature profiles and identify observations that do not fit into any cluster.

**Process overview:** The workflow selects numerical features, handles missing values, scales the data, and evaluates multiple DBSCAN parameter combinations using the silhouette score. The best parameter set is then used to fit DBSCAN and flag noise points as outliers.

**Method note:** DBSCAN groups observations in dense regions and labels low-density observations as noise. The `eps` parameter controls neighborhood distance, while `min_samples` controls the minimum number of observations required to form a dense region.

In [ ]:
# Density-Based Spatial Clustering of Applications with Noise (DBSCAN)

import warnings
warnings.filterwarnings('ignore')

# Assuming 'df' is your DataFrame from the previous analysis
# Select numerical features for analysis (exclude 'image file' and any non-numeric columns)
numeric_cols = df.select_dtypes(include=[np.number]).columns
features = df[numeric_cols]

# Handle missing values: Remove rows with NaN values in the features
features_clean = features.dropna()
df_clean = df.loc[features_clean.index]

# Standardize the features using StandardScaler
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features_clean)

# Automate parameter tuning for DBSCAN
# Define parameter grid
eps_values = np.linspace(0.1, 5.0, 50)  # Adjust the range and steps as needed
min_samples_values = range(2, 10)

best_score = -1
best_params = {'eps': None, 'min_samples': None}
best_labels = None

print("Starting parameter tuning for DBSCAN...")

for eps, min_samples in product(eps_values, min_samples_values):
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(features_scaled)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

    # Only consider cases with at least 2 clusters and less than the total number of samples
    if n_clusters > 1 and n_clusters < len(features_scaled):
        # Compute the silhouette score (requires at least 2 clusters)
        try:
            score = silhouette_score(features_scaled, labels)
            if score > best_score:
                best_score = score
                best_params['eps'] = eps
                best_params['min_samples'] = min_samples
                best_labels = labels
        except Exception:
            continue

print(f"\nBest parameters found:")
print(f"eps: {best_params['eps']}")
print(f"min_samples: {best_params['min_samples']}")
print(f"Silhouette Score: {best_score}")

# Fit DBSCAN with best parameters
dbscan = DBSCAN(eps=best_params['eps'], min_samples=best_params['min_samples'])
dbscan.fit(features_scaled)

# Add cluster labels to the DataFrame
df_clean['DBSCAN_Cluster'] = dbscan.labels_
df_clean['Outlier_DBSCAN'] = df_clean['DBSCAN_Cluster'] == -1  # True if outlier

# Display outliers
outliers_dbscan = df_clean[df_clean['Outlier_DBSCAN']]
print("\nOutliers detected using DBSCAN with best parameters:")
print(outliers_dbscan[['image file', 'DBSCAN_Cluster']])

# Optionally, merge the outlier information back into the original DataFrame
df['DBSCAN_Cluster'] = df_clean['DBSCAN_Cluster']
df['Outlier_DBSCAN'] = df_clean['Outlier_DBSCAN']

## 7. Ensemble Outlier Scoring

**Purpose:** Combine model-specific anomaly flags into a single consensus outlier result.

**Process overview:** The workflow converts each model's outlier flag to a consistent boolean format, sums the number of methods that flagged each observation, and marks observations as ensemble outliers when they meet the selected agreement threshold. The final outlier results are displayed and saved to a CSV file.

**Method note:** The ensemble approach balances methods with different assumptions, including density-based detection, reconstruction-error detection, boundary-based detection, and isolation-based detection. Requiring agreement across methods helps prioritize images with stronger evidence of unusual visual patterns.

In [ ]:
# Ensemble scoring based on all anomoly detection methods above

# Ensure all outlier flags are boolean
df['Outlier_IsolationForest'] = df['Outlier_IsolationForest'].astype(bool)
df['Outlier_Autoencoder'] = df['Outlier_Autoencoder'].astype(bool)
df['Outlier_LOF'] = df['Outlier_LOF'].astype(bool)
df['Outlier_OCSVM'] = df['Outlier_OCSVM'].astype(bool)
df['Outlier_DBSCAN'] = df['Outlier_DBSCAN'].astype(bool)

# Sum the number of times each observation is flagged as an outlier
df['Outlier_Sum'] = (
    df['Outlier_IsolationForest'].astype(int) +
    df['Outlier_Autoencoder'].astype(int) +
    df['Outlier_LOF'].astype(int) +
    df['Outlier_OCSVM'].astype(int) +
    df['Outlier_DBSCAN'].astype(int)
)

# Decide on a threshold
threshold = 2

# Create the ensemble outlier flag
df['Outlier_Ensemble'] = df['Outlier_Sum'] >= threshold

# Display the observations flagged as outliers by the ensemble method
outliers_ensemble = df[df['Outlier_Ensemble']]
print("Outliers detected by Ensemble Method:")
print(outliers_ensemble[['image file', 'Outlier_Sum']])

# Ensure the image directory exists
if not os.path.exists(image_dir):
    raise FileNotFoundError(f"The specified image directory does not exist: {image_dir}")

# Save the updated DataFrame to a CSV file in the image directory
output_file_path = os.path.join(image_dir, 'image_analysis_with_ensemble_outliers.csv')
df.to_csv(output_file_path, index=False)
print(f"\nEnsemble analysis results saved to '{output_file_path}'")